UPLOADING THE DEMO SUBSET

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

subset_demo = "horoscope_demo"
df = pd.read_csv(f'..\\data\\{subset_demo}.csv')
df.head()

,ID,sign,category,date,horoscope
0,18,aries,general,20200704,You may have found yourself having some issues...
1,22,aries,general,20200708,You'll find little comfort in your emotions to...
2,51,aries,general,20200806,"Your mind is buzzing like a bee, Aries. You're..."
3,63,aries,general,20200818,This is an excellent day to express your natur...
4,72,aries,general,20200827,"All things domestic are highlighted, Aries. It..."


1. MINHASHING + LSH

1.1 Normalization function and shingling function

In [3]:
import re

def normalize_text(text):
    text = text.lower()
    
    # 1. Remove punctuation and replace with a space
    text = re.sub(r"[^\w\s]", " ", text) # Removes everything that is not a word character or whitespace
    
    # 2. Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # 3. Return the list of words (tokens)
    return text.split(' ')


def shingle(q, text):
    newtext = normalize_text(text)
    shingles = []
    # The index must go from 0 to the last shingle index. 
    # The last shingle index is N (number of words) - q. 
    # We add +1 so it's not excluded by the Python for loop's range.
    for i in range(len(newtext) - q + 1): 
        # The join method combines elements of a list into a single string, 
        # separated by a space character (" "). In this case, 
        # the list elements go from index i to index i+q.
        shingles.append(" ".join(newtext[i:i + q])) 
    
    # Remove repetitions
    unique = set(shingles) 
    # Convert back to a list
    list_shingles = list(unique) 
    
    return list_shingles

1.2 Function to hash a list of strings (taken from excercises about minhashing)

In [4]:
import sys
import os
import mmh3


#################### Utilities ######################
#hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ mmh3.hash(e, seed)
	return val 

1.3 Signature function with the use of SIG - Matrix to store and update all the signatures together while I'm sliding the shingles

In [5]:
def signature(docs: dict, seedlist: list, q: int):
    
    
    doc_ids = list(docs.keys()) #extract dictionary id from dict, now I have a list with dictionary id
    k = len(seedlist)# k is the number of hash functions
    
    
    shingle_to_docs = {} #prepare an empty dict
    
   
  # Initial loop to populate the index and prepare the shingles
    for doc_id, text in docs.items(): #associate in doc_id the ID of the documents and in text the text of the documents
        
        shingles_list = shingle(q, text)#I prepare the list of shingles, this is for a document
        for shingle_item in shingles_list:
          
            shingle_to_docs.setdefault(shingle_item, set()).add(doc_id)

        # I need the double loop to scroll through all the documents and every shingle in each document
        # I have my empty list to which I apply the setdefault function this function takes the key (a shingle)
        # and searches for it in the dictionary if there is one, adds another ID document to the value set collection, 
        # if not there create this key and add the document ID immediately inside a set object
        # what I have at the end is a big dictionary with all the shingles as key and the documents where they are present as value
         

    doc_signatures = {              #I define the signature dictionary with id as key and a list as long as the hash functions with only inf inside, so k inf
        doc_id: [float("inf")] * k  #[float("inf")] * k when you multiply a list containing a single element by an integer, the result
                                    #is a new list in which that element is repeated k times.
        for doc_id in doc_ids
    } # so now I have a dict initialized with the keys of the documents and as value a list as long as the number of f.ash of infinity
    
    # at this point I have a dict that has shingles as key and the list where the shingle is located as value shingle_to_docs
    # I also have a dict doc_signatures which has as key the IDs of the documents and as value instead a list of infinite numbers as many as the hash functions
    
    for shingle_item in shingle_to_docs.keys():# get the shingle (key) from the shingle list (cycles through all shingles)
        
        hash_values = [listhash(shingle_item, seed) for seed in seedlist] # I take the shingle and apply all the hash functions to it and save them in a list
        
        for doc_id in shingle_to_docs[shingle_item]: #scroll through all the IDs where you find this shingle. shingle_to_docs[shingle_item] is the value or the ID list where you can
                                                     #find the shingle and scroll through all the IDs
            current_signature = doc_signatures[doc_id]#the current signature is first all infinities and then updates with all minima
            for i in range(k):
                current_signature[i] = min(current_signature[i], hash_values[i])
                
    #I take the shingle and apply all the hash functions to it and save them in a list that will have length k
    #get the list of documents where this shingle is present
    # for each document in which it is present I look for its current signature
    # function by function check if the ash I have now is smaller in case I replace
    # then for every shingles, for every document it is contained in and for every ash function.           
    return doc_signatures

1.4 Jaccard similarity function: comparing for each position the items inside the signatures

In [6]:
import numpy as np
def jaccard (doc_id_1: str, doc_id_2: str, doc_signatures: dict):
    sign1 = np.array(doc_signatures[doc_id_1])
    sign2 = np.array(doc_signatures[doc_id_2])
    matches = 0
    k = len(sign1)
    
    for i in range(k):
        if sign1[i] == sign2[i]:
            matches += 1
            
    similarity = matches / k
    return similarity

1.5 LSH function (I have also a similarity function in the original notbeook but I'm using only the lsh since I have around 17500 texts)

In [7]:
def lsh(signatures_dict, b, jaccard_threshold=0.5, seed=42):
    lsh_dict = {} # new dict that has as key the IDs and as value the hashes of the blocks
    for key, values in signatures_dict.items(): # for each item with its key value ID: signature
        blocks = np.split(np.array(values), b) # split the signature into blocks
        blocks_hash_values = [] # empty list for the new signature
        for aBlock in blocks: # for each block among the blocks
            band_bytes = aBlock.tobytes()
            # hash for each block until a list of hashes is created
            blocks_hash_values.append(mmh3.hash(band_bytes, seed)) 
        # in the dict I put the ID in the key and the new list of hashed blocks in the value
        lsh_dict[key] = blocks_hash_values 
        
    list_keys = list(lsh_dict.keys()) # I save the list of keys
    similar_items = {} # new dict
    
    for i in range (len(list_keys)-1):
        for j in range (i+1, len(list_keys)):
            # how many in common in the new list?
            common_values = np.intersect1d(lsh_dict[list_keys[i]], lsh_dict[list_keys[j]]) 
            
            # if at least one then they are candidates and I calculate them with jaccard
            if len(common_values) > 0: 
                # we found a candidate
                similarity_score = jaccard(list_keys[i], list_keys[j], signatures_dict)
                
                # if they exceed the threshold
                if similarity_score >= jaccard_threshold: 
                    # the key of similar items are the name of the two documents and the value is the similarity
                    similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                    
    return similar_items

1.6 Function to count pairs after lsh

In [8]:
def counter_pair_sim(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim } 
    length_dict = len(pair_sim)
    return length_dict

1.7 Function to find how many horoscopes in one category are involved in at least one couple

In [9]:
def oroscope_least1(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim }
    horoscope_least_1_sim = set()
    for pair in pair_sim.keys():
        horoscope_least_1_sim.update(pair)
    return len(horoscope_least_1_sim)

1.8 Excluding Birthday (The reason is explained in the report) 

In [10]:
# Create text ID dictionary
df_full = df[df['category']!= 'birthday']
horoscope_full_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_full.iterrows()  
}

# Example: first 40 elements
print(list(horoscope_full_dict.items())[:40])  

[(18, "You may have found yourself having some issues with food lately, Aries. It could be that your sense of self-worth isn't at its highest and you're trying to make up for it by sabotaging your relationship with your body. Food is healthy nourishment that you need to survive. Your body deserves respect. You need to always give it the proper fuel that it requires to be healthy."), (22, "You'll find little comfort in your emotions today, Aries. You may want to simply stick to business. Concentrate on getting things done in your regular routine. Create a plan and stick to it. This isn't a day to deviate from the norm, nor is it a time in which you'll find sympathy from others. Stick close to home and take care of your personal business. Time is precious - don't waste it."), (51, "Your mind is buzzing like a bee, Aries. You're apt to find that there's very little you can do to slow it down. Make sure you add compassion to the chain that's holding everything together. Also make sure that

1.9 Apply the LSH

In [11]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_full_dict, seedlist, q)

# Find similar items using the LSH function
full_similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)
print('couples with similarity >= 0.3', len(full_similar_items_lsh))


couples with similarity >= 0.3 51


1.10 Print Results

In [12]:
for (id1, id2), sim in full_similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_full_dict[id1]}")
    print(f"Text2: {horoscope_full_dict[id2]}")
    print("------")

ID1: 754, ID2: 21066, Similarità: 1.00
Text1: Close the door on old projects so you can make way for new ones. You have many loose ends right now that are causing mental clutter whether you realize it or not. Delegate your work, tackle it yourself with fresh eyes, or toss it out completely.
Text2: Close the door on old projects so you can make way for new ones. You have many loose ends right now that are causing mental clutter whether you realize it or not. Delegate your work, tackle it yourself with fresh eyes, or toss it out completely.
------
ID1: 936, ID2: 2584, Similarità: 1.00
Text1: Confusion will arise around mid-day and will most likely set you back if you are not prepared to deal with it. Set your plan during the beginning of the day and then use the afternoon to follow through. Plans initiated mid-day will most likely flop.
Text2: Confusion will arise around mid-day and will most likely set you back if you are not prepared to deal with it. Set your plan during the beginning 

1.11 Counting how many couples for certain similarity tresholds

In [13]:
print("dataset lenght", len(horoscope_full_dict))
print("\n")

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(full_similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 1440


Number of couples with similarity ≥ 1 :  51 

Number of couples with similarity ≥ 0.9 :  51 

Number of couples with similarity ≥ 0.75 :  51 

Number of couples with similarity ≥ 0.5 :  51 

Number of couples with similarity ≥ 0.3 :  51 



1.12 Saving the Dataframe

In [14]:
series = pd.Series(full_similar_items_lsh)


df_full_similar_items_lsh = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

df_full_similar_items_lsh['Similarity'] = series.values

print("## final table")
print(df_full_similar_items_lsh)
# df.to_csv('similarity_full_lsh_DEMO.csv', index=False)

## final table
      ID1    ID2  Similarity
0     754  21066         1.0
1     936   2584         1.0
2    1120  17860         1.0
3    1149  17889         1.0
4    1217  19697         1.0
5    1390   6610         1.0
6    1397   4877         1.0
7    1419   6639         1.0
8    2766   4414         1.0
9    2950  19690         1.0
10   2979  19719         1.0
11   3047   6676         1.0
12   3047  21527         1.0
13   3204  21535         1.0
14   3220   8440         1.0
15   3227   6707         1.0
16   3249   8469         1.0
17   4596   6244         1.0
18   4780  21520         1.0
19   4809  21549         1.0
20   5050  10270         1.0
21   5057   8537         1.0
22   5079  10299         1.0
23   6426   8074         1.0
24   6676  21527         1.0
25   6880  12100         1.0
26   6887  10367         1.0
27   6909  12129         1.0
28   8256   9904         1.0
29   8710  13930         1.0
30   8717  12197         1.0
31   8739  13959         1.0
32  10086  11734         1.0

1.13 Assesing how many couples have two different categories

In [15]:
df_full = df_full_similar_items_lsh

# Create an ID category dictionary from the full dataset
ID_CATEGORY = df.set_index('ID')['category'].to_dict() #here I can use .to_dict() since i'm not excluding anything


# Calculate the sum of differences directly by iterating through the DataFrame by index.
# Compare ID_CATEGORY[id1] with ID_CATEGORY[id2] 
num_diff_cat = sum([1 for i in range(len(df_full)) 
                    if ID_CATEGORY[df_full['ID1'][i]] != ID_CATEGORY[df_full['ID2'][i]]])


print("Number of pairs with different categories:", num_diff_cat)

# Percentage relative to the total number of pairs
perc_diff_cat = num_diff_cat / len(df_full) * 100
print("Percentage of pairs with different categories: {:.2f}%".format(perc_diff_cat))

Number of pairs with different categories: 0
Percentage of pairs with different categories: 0.00%


1.14 from here the analysis continues splitting the categories as explained in the report. The code and the analysis is exactly the same for each category. Here it will be presented just an example with the category WELLNESS (go to the original notebook if you are interesed in the other categories)

1.15 WELLNESS Category

In [16]:
df_wellness = df[df["category"] == "wellness"]

horoscope_wellness_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_wellness.iterrows()
}

1.16 Applying LSH

In [17]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_wellness_dict, seedlist, q)

# Find similar items using the LSH function

wellness_similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)

1.17 Print results

In [18]:
# Print results
for (id1, id2), sim in wellness_similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_wellness_dict[id1]}")
    print(f"Text2: {horoscope_wellness_dict[id2]}")
    print("------")

ID1: 1120, ID2: 17860, Similarità: 1.00
Text1: A lot of emphasis is put on diet these days - choosing the freshest foods available and preparing a variety of healthy meals so that all your nutritional bases are covered is key. Another important part of diet is to eat slowly and chew your food carefully. Apart from avoiding the risk of choking, well-chewed food helps us avoid over-taxing our digestive glands. Many people who gulp down their food over-work their glands and find that they stay "hungry" even after eating!
Text2: A lot of emphasis is put on diet these days - choosing the freshest foods available and preparing a variety of healthy meals so that all your nutritional bases are covered is key. Another important part of diet is to eat slowly and chew your food carefully. Apart from avoiding the risk of choking, well-chewed food helps us avoid over-taxing our digestive glands. Many people who gulp down their food over-work their glands and find that they stay "hungry" even after 

1.18 Counting how many coupes for a certain treshsold

In [19]:
print("dataset lenght", len(horoscope_wellness_dict))
print("\n")

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(wellness_similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 360


Number of couples with similarity ≥ 1 :  39 

Number of couples with similarity ≥ 0.9 :  39 

Number of couples with similarity ≥ 0.75 :  39 

Number of couples with similarity ≥ 0.5 :  39 

Number of couples with similarity ≥ 0.3 :  39 



1.19 Counting the lower limit percentage 

In [20]:
for t in thresholds:
    print("horoscope rycicled at least once with similarity >=", t, ": ",  oroscope_least1(wellness_similar_items_lsh, t))
    print("lower limit percentage of recycled dataset >=", t, ": ",  (oroscope_least1(wellness_similar_items_lsh, t)/2)/len(horoscope_wellness_dict)*100)
    print("\n")

horoscope rycicled at least once with similarity >= 1 :  75
lower limit percentage of recycled dataset >= 1 :  10.416666666666668


horoscope rycicled at least once with similarity >= 0.9 :  75
lower limit percentage of recycled dataset >= 0.9 :  10.416666666666668


horoscope rycicled at least once with similarity >= 0.75 :  75
lower limit percentage of recycled dataset >= 0.75 :  10.416666666666668


horoscope rycicled at least once with similarity >= 0.5 :  75
lower limit percentage of recycled dataset >= 0.5 :  10.416666666666668


horoscope rycicled at least once with similarity >= 0.3 :  75
lower limit percentage of recycled dataset >= 0.3 :  10.416666666666668




1.20 Saving the results

In [21]:
series = pd.Series(wellness_similar_items_lsh)

df_well = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

df_well['Similarity'] = series.values

print("## final table")
print(df_well)
# df.to_csv('similarity_wellness_lsh.csv', index=False)

## final table
      ID1    ID2  Similarity
0    1120  17860         1.0
1    1149  17889         1.0
2    1217  19697         1.0
3    1390   6610         1.0
4    1397   4877         1.0
5    1419   6639         1.0
6    2950  19690         1.0
7    2979  19719         1.0
8    3047   6676         1.0
9    3047  21527         1.0
10   3204  21535         1.0
11   3220   8440         1.0
12   3227   6707         1.0
13   3249   8469         1.0
14   4780  21520         1.0
15   4809  21549         1.0
16   5050  10270         1.0
17   5057   8537         1.0
18   5079  10299         1.0
19   6676  21527         1.0
20   6880  12100         1.0
21   6887  10367         1.0
22   6909  12129         1.0
23   8710  13930         1.0
24   8717  12197         1.0
25   8739  13959         1.0
26  10540  15760         1.0
27  10547  14027         1.0
28  10569  15789         1.0
29  12370  17590         1.0
30  12377  15857         1.0
31  12399  17619         1.0
32  14200  19420         1.0

1.21 Intrasign couples Analysis

Function to count the same sign pairs

In [22]:
def count_same_sign_pairs(df_horoscope, df_couples, category_name=""):
    
    ID_SIGN = df_horoscope.set_index('ID')['sign'].to_dict() #I CAN USE .to_dict() since tha dataframe was already filtered before

    
    df_couple_dict = {
        (row['ID1'], row['ID2']): {
            'similarity': row['Similarity'],
            'sign1': ID_SIGN[row['ID1']],
            'sign2': ID_SIGN[row['ID2']]
        }
        for _, row in df_couples.iterrows()
    }

    # Counting same sign
    same_sign_count = 0
    for item in df_couple_dict.values():
        if item['sign1'] == item['sign2']:
            same_sign_count+=1

    # Percentage
    percentage = (same_sign_count / len(df_couples))*100

    # Output readable
    print("Category:", category_name)
    print("Number of couples with the same sign:", same_sign_count)
    print("Percentage:", percentage)

    return same_sign_count, percentage, df_couple_dict

1.22 Applying the function

In [24]:
same_count, perc, wel_dict = count_same_sign_pairs(df, df_well, "wellness")

Category: wellness
Number of couples with the same sign: 0
Percentage: 0.0
